
# AI IN HEALTHCARE: High Risk Project COLAB NOTEBOOK
# Mohsin Imam
# Topic: Translator


In [12]:
!pip install -q -U google-generativeai pandas

In [3]:
import pandas as pd
import google.generativeai as genai
import json
import time
from google.colab import userdata

# --- 1. SETUP GEMINI API ---
genai.configure(
    api_key=userdata.get('GEMINI_API_KEY'),
    client_options={"api_endpoint": "generativelanguage.googleapis.com"} # Explicitly set API endpoint
)
# We use JSON mode to ensure the output is perfectly structured for our training dataset
model = genai.GenerativeModel(
    'gemini-2.5-flash',
    generation_config={"response_mime_type": "application/json"}
)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [9]:
# --- 2. LOAD SYNTHEA DATA ---
print("Loading Synthea CSVs...")
try:
    patients_df = pd.read_csv('patients.csv')
    conditions_df = pd.read_csv('conditions.csv')
    medications_df = pd.read_csv('medications.csv')
except FileNotFoundError:
    print("Error: Please make sure patients.csv, conditions.csv, and medications.csv are uploaded to Colab.")

# --- 3. STITCH TABULAR DATA INTO CLINICAL NOTES ---
print("Generating synthetic clinical notes...")
patient_ids = patients_df['Id'].head(5).tolist()
synthetic_notes = []

for pid in patient_ids:
    # Extract unique conditions and medications for the specific patient
    pt_conditions = conditions_df[conditions_df['PATIENT'] == pid]['DESCRIPTION'].dropna().unique().tolist()
    pt_meds = medications_df[medications_df['PATIENT'] == pid]['DESCRIPTION'].dropna().unique().tolist()

    # Skip patients with no medical history to ensure our training data is robust
    if not pt_conditions:
        continue

    # Construct the jargon-heavy note
    note = f"Patient ID {pid[:8]} presents with a documented medical history significant for {', '.join(pt_conditions)}. "
    if pt_meds:
        note += f"Current pharmacological interventions include: {', '.join(pt_meds)}. "
    note += "Plan: Continue current care regimen, monitor for therapeutic efficacy and adverse interactions. Follow up in outpatient clinic."

    print(note)
    synthetic_notes.append(note)

notes_df = pd.DataFrame({'clinical_note': synthetic_notes})
print(f"Successfully generated {len(notes_df)} clinical notes.\n")

# --- 4. GENERATE GROUND TRUTH (TEACHER MODEL) ---
print("Calling Gemini API to generate English & Urdu ground truth...")
training_data = []

# We'll process a small batch first to ensure everything works perfectly
for index, row in notes_df.iterrows():
    print(f"Translating note {index + 1}/{len(notes_df)}...")

    prompt = f"""
    You are an expert medical translator. Read the following clinical note and output exactly two things:
    1. "simple_english": A simplified summary written at an 8th-grade reading level, removing complex jargon but keeping clinical accuracy.
    2. "urdu_translation": A highly accurate, culturally appropriate Urdu translation of the simplified English summary.

    Clinical Note: {row['clinical_note']}
    """

    try:
        response = model.generate_content(prompt, request_options={"timeout": 60}) # Added timeout
        print(f"API Response for note {index + 1}: {response.text[:100]}...") # Log partial response
        result = json.loads(response.text)

        # Structure it exactly how the Gemini Fine-Tuning API expects it
        training_data.append({
            "text_input": row['clinical_note'],
            "output": f"Simple English: {result['simple_english']}\n\nUrdu: {result['urdu_translation']}"
        })

        # Sleep to respect API rate limits
        time.sleep(3)

    except Exception as e:
        print(f"Error processing note {index + 1}: {e}")

# --- 5. SAVE DATASET ---
# Convert the training data dictionary into a DataFrame to inspect and save
training_df = pd.DataFrame(training_data)
training_df.to_csv('fine_tuning_dataset.csv', index=False)

print("\nPipeline Complete! Here is a preview of your training data:")
print(training_df.head())

Loading Synthea CSVs...
Generating synthetic clinical notes...
Patient ID f1aa52b9 presents with a documented medical history significant for Housing unsatisfactory (finding), Received higher education (finding), Loss of teeth (disorder), Full-time employment (finding), Acute bronchitis (disorder), Medication review due (situation), Unemployed (finding), Viral sinusitis (disorder), Reports of violence in the environment (finding), Stress (finding), Acute viral pharyngitis (disorder), Gingivitis (disorder), Gingival disease (disorder). Current pharmacological interventions include: Acetaminophen 325 MG Oral Tablet, Amoxicillin 250 MG / Clavulanate 125 MG Oral Tablet, sodium fluoride 0.0272 MG/MG Oral Gel. Plan: Continue current care regimen, monitor for therapeutic efficacy and adverse interactions. Follow up in outpatient clinic.
Patient ID d30ace70 presents with a documented medical history significant for Received higher education (finding), Prediabetes (finding), Stress (finding), B

In [14]:
!pip install -q -U transformers peft trl bitsandbytes datasets kernels

In [16]:
pip install --upgrade pyarrow pandas datasets

In [1]:
import pandas as pd
from datasets import Dataset

# 1. Load the ground-truth dataset we generated earlier
df = pd.read_csv('fine_tuning_dataset.csv')

# 2. Define the Llama-3 chat template format
def format_llama_prompt(row):
    """Wraps our inputs and outputs in Llama-3's official chat format."""
    system_prompt = "You are an expert medical translator. Your job is to translate complex clinical notes into simple English and accurate Urdu."

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>
{row['text_input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
{row['output']}<|eot_id|>"""

    return {"text": prompt}

# 3. Convert the Pandas DataFrame into a Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

# Apply the formatting function to every row
hf_dataset = hf_dataset.map(format_llama_prompt)

print(f"Prepared {len(hf_dataset)} examples for training.")
print("\nPreview of the first formatted prompt:")
print(hf_dataset[0]['text'])

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Prepared 5 examples for training.

Preview of the first formatted prompt:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert medical translator. Your job is to translate complex clinical notes into simple English and accurate Urdu.<|eot_id|><|start_header_id|>user<|end_header_id|>
Patient ID f1aa52b9 presents with a documented medical history significant for Housing unsatisfactory (finding), Received higher education (finding), Loss of teeth (disorder), Full-time employment (finding), Acute bronchitis (disorder), Medication review due (situation), Unemployed (finding), Viral sinusitis (disorder), Reports of violence in the environment (finding), Stress (finding), Acute viral pharyngitis (disorder), Gingivitis (disorder), Gingival disease (disorder). Current pharmacological interventions include: Acetaminophen 325 MG Oral Tablet, Amoxicillin 250 MG / Clavulanate 125 MG Oral Tablet, sodium fluoride 0.0272 MG/MG Oral Gel. Plan: Continue current care regimen, mon

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from huggingface_hub import login
from google.colab import userdata

# Authenticate with Hugging Face
login(token=userdata.get('HF_TOKEN'))

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# 1. Configure 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading Tokenizer and Model in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fixes an issue with mixed-precision training

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 2. Configure LoRA (Low-Rank Adaptation)
lora_config = LoraConfig(
    r=16, # Rank of the adapter
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"], # Target the attention mechanisms
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach the LoRA adapters to the base model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading Tokenizer and Model in 4-bit...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


In [3]:
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Define Training Hyperparameters
training_args = TrainingArguments(
    output_dir="./llama3-health-translator",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit", # Optimizer optimized for low-memory
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100, # Adjust this based on your dataset size (e.g., 100-300 steps is usually enough for a small dataset)
    fp16=False, # Mixed precision for faster training
    save_strategy="epoch",
    gradient_checkpointing=False # Disabling gradient checkpointing to resolve Params4bit detach issue
)

# 2. Initialize the Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    args=training_args,
)

print("Starting the training loop...")
# 3. KICK OFF TRAINING
trainer.train()

print("\nTraining Complete! Saving model...")
# 4. Save the fine-tuned LoRA weights
trainer.model.save_pretrained("final_health_translator_adapters")

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Starting the training loop...


Step,Training Loss
10,1.366109
20,0.800847
30,0.319172
40,0.090967
50,0.041585
60,0.030575
70,0.027036
80,0.025931
90,0.025570
100,0.025348



Training Complete! Saving model...
